# 🏀 Proyecto Machine Learning - Draft NBA 2026 🏀

## ¿Qué es eso del Draft de la NBA?

El **Draft de la NBA** es el proceso anual mediante el cual los 30 equipos de la NBA seleccionan jugadores de universidad, ligas europeas y otras competiciones. Consta de 2 rondas y 60 picks.

La posición en la que eres drafteado determina tu contrato, tu equipo y en muchos casos tu carrera entera.

---
## Punto de partida

Cada año, los equipos NBA seleccionan jugadores en el Draft. Las predicciones tradicionales se basan en opiniones de analistas o de profesionales dedicados a seguir la trayectoria de jóvenes talentos. Sin embargo, me he querido preguntar:

¿Podemos **predecir con Machine Learning** en qué **posición será drafteado un jugador** basándonos en sus estadísticas universitarias y sus medidas físicas? En concreto, ¿en qué ronda y posición saldrán seleccionados los españoles Aday Mara, Baba Miller y Sergio De Larrea?

---
## Fundamentos del proyecto

### 1. Definir las predicciones

- ¿En qué **ronda** será drafteado un jugador? 1ª ronda / 2ª ronda / no drafteado (modelo de clasificación)
- ¿En qué **pick** exacto caerá? (modelo de regresión, solo para jugadores predichos como drafteados)

### 2. ¿Por qué es interesante este proyecto?

Las predicciones actuales se basan en **opiniones subjetivas** de scouts y analistas. Este proyecto propone un enfoque basado en **datos objetivos**: estadísticas reales de rendimiento en cancha y medidas físicas de cada jugador.

### 3. ¿A quién le interesa?

Además de al propio aficionado al baloncesto, también puede llamar la atención de un **medio de comunicación deportivo** que quiere publicar un artículo sobre las posibilidades de los jugadores españoles en el Draft 2026 respaldado no por opiniones, sino por un modelo de Machine Learning. Algo distinto y muy actual con el auge de la IA.

---

---
## Fuentes de datos

### Dataset 1 — Estadísticas universitarias NCAA (2009–2021)
- Más de 61.000 registros de jugadores de la NCAA entre 2009 y 2021.
- Una fila por jugador por temporada.
- Variables: puntos, rebotes, asistencias, tapones, robos, eficiencia de tiro (eFG%, TS%), BPM, usage rate y muchas más.
- **Incluye directamente el número de pick** del Draft para los jugadores seleccionados — es nuestra variable objetivo.
- Fuente: Kaggle (`College_BasketballPlayers2009-2021`).

### Dataset 2 — NBA Draft Combine histórico (2000–2026)
- Jugadores que participaron en el Combine entre los años 2000 y 2026.
- Variables: medidas físicas (altura, envergadura, peso, salto vertical, agilidad, sprint...).
- Obtenido mediante la librería oficial de la NBA para Python (`nba_api`).
- Incluye los datos de los tres españoles de cara a la predicción final.

### ¿Por qué estos dos datasets?

El dataset de la NCAA nos da las **variables de rendimiento** (lo que hace el jugador en cancha) y la **variable objetivo** (qué pick tuvo en el Draft).  
El Combine nos da las **variables físicas** (cuerpo y atletismo puro).  
Cruzamos ambos por nombre para construir un dataset completo con las dos dimensiones que valoran los equipos de la NBA.

---

In [ ]:
import pandas as pd

ncaa = pd.read_csv('College_BasketballPlayers2009-2021.csv', low_memory=False)
combine = pd.read_csv('combine_historico.csv')

print(f"NCAA histórico: {ncaa.shape[0]:,} registros | {ncaa.shape[1]} variables")
print(f"Combine histórico: {combine.shape[0]:,} jugadores | {combine.shape[1]} variables")
print()
print(f"Jugadores drafteados en NCAA con pick conocido: {ncaa['pick'].notna().sum():,}")
print(f"Jugadores no drafteados en NCAA: {ncaa['pick'].isna().sum():,}")

---
## Los tres españoles en el Combine 2026

Los tres candidatos que queremos predecir participaron en el NBA Draft Combine 2026, donde se midieron sus capacidades físicas y atléticas.

| Jugador | Posición | Altura (sin zapatillas) | Peso | Envergadura |
|---------|----------|------------------------|------|-------------|
| Aday Mara | C | 221 cm | 117.8 kg | 229 cm |
| Baba Miller | PF | 209 cm | 94.4 kg | 218 cm |
| Sergio De Larrea | PG | 198 cm | 79 kg | 202 cm |

> **Nota:** Sergio De Larrea figura con datos incompletos en el Combine oficial. Sus medidas se completarán manualmente a partir de fuentes externas antes de realizar la predicción.

In [ ]:
# Verificar que los tres españoles están en el combine 2026
combine['HEIGHT_WO_SHOES'] = pd.to_numeric(combine['HEIGHT_WO_SHOES'], errors='coerce')
combine['WEIGHT'] = pd.to_numeric(combine['WEIGHT'], errors='coerce')
combine['WINGSPAN'] = pd.to_numeric(combine['WINGSPAN'], errors='coerce')

espanoles = combine[
    (combine['SEASON'] == 2026) &
    (combine['LAST_NAME'].isin(['Mara', 'Miller', 'De Larrea']))
][['PLAYER_NAME', 'POSITION', 'HEIGHT_WO_SHOES', 'WEIGHT', 'WINGSPAN',
   'MAX_VERTICAL_LEAP', 'LANE_AGILITY_TIME', 'THREE_QUARTER_SPRINT']]

espanoles

---

## Construcción del dataset de entrenamiento

Para entrenar el modelo, cruzamos los dos datasets por nombre de jugador:

- **Variables de cancha** → del dataset NCAA: puntos, rebotes, asistencias, eficiencia...
- **Variables físicas** → del Combine: altura, peso, envergadura, salto vertical, sprint...
- **Variable objetivo** → la columna `pick` del dataset NCAA (número de elección en el Draft)

Para los jugadores **no drafteados** que pasaron por el Combine, los incluimos como clase `ND` (no drafteado), lo que nos permite entrenar también el modelo de clasificación de ronda.

In [ ]:
# Merge: NCAA + Combine por nombre de jugador
ncaa['name_clean'] = ncaa['player_name'].str.strip().str.lower()
combine['name_clean'] = combine['PLAYER_NAME'].str.strip().str.lower()

combine_hist = combine[(combine['SEASON'] >= 2009) & (combine['SEASON'] <= 2021)].copy()

# Features físicas del combine
cols_fisicas = ['name_clean', 'SEASON', 'POSITION', 'HEIGHT_WO_SHOES', 'WEIGHT',
                'WINGSPAN', 'STANDING_REACH', 'STANDING_VERTICAL_LEAP',
                'MAX_VERTICAL_LEAP', 'LANE_AGILITY_TIME', 'THREE_QUARTER_SPRINT']

# Features de cancha del NCAA
cols_cancha = ['name_clean', 'year', 'pick', 'pts', 'ast', 'treb', 'stl', 'blk',
               'bpm', 'obpm', 'dbpm', 'eFG', 'TS_per', 'usg',
               'TP_per', 'FT_per', 'Ortg', 'drtg', 'AST_per', 'TO_per',
               'blk_per', 'stl_per', 'ORB_per', 'DRB_per']

df = pd.merge(
    ncaa[cols_cancha],
    combine_hist[cols_fisicas],
    on='name_clean',
    how='inner'
)

# Filtrar: el combine debe ser del mismo año o el siguiente al de las stats NCAA
df['year_diff'] = df['SEASON'].astype(int) - df['year'].astype(int)
df = df[df['year_diff'].isin([0, 1])].copy()

# Crear target de clasificación
df['ronda'] = df['pick'].apply(lambda x: 'R1' if x <= 30 else ('R2' if x <= 60 else 'ND'))
df['ronda'] = df['ronda'].fillna('ND')

print(f"Dataset de entrenamiento: {len(df):,} jugadores")
print()
print("Distribución de clases (clasificación):")
print(df['ronda'].value_counts())

---

## Modelos planificados

El proyecto contempla tres enfoques complementarios:

### 1. Clasificación — ¿En qué ronda será drafteado?
**Target:** 1ª ronda / 2ª ronda / no drafteado  
**Algoritmo:** XGBoost / LightGBM  
**Features:** estadísticas NCAA + medidas físicas del Combine

### 2. Regresión — ¿Qué pick exacto?
**Target:** número de pick (1–60)  
**Condición:** solo para jugadores cuya clasificación sea R1 o R2  
**Algoritmo:** XGBoost con optimización de hiperparámetros

### 3. No supervisado — ¿A qué perfil pertenece? *(idea a desarrollar)*

Se agrupa automáticamente a todos los jugadores históricos en clusters según sus medidas físicas y estadísticas de rendimiento. Por ejemplo: *pívots dominantes tipo Jokic*, *bases anotadores*, *aleros atléticos y defensivos*, etc. Aday Mara podría encajar en el cluster de pívots europeos con gran envergadura, como referencia al tipo de impacto esperado en la NBA.

---

## ¿Cuál es el objetivo final?

- Un **análisis completo** sobre las posibilidades reales de Aday Mara, Baba Miller y Sergio De Larrea en el Draft 2026, respaldado por Machine Learning.
- Crear una **app interactiva** donde introducir las estadísticas de cualquier jugador y obtener una predicción de su pick en el Draft.
- Acercar el mundo del periodismo deportivo a herramientas tecnológicas poco utilizadas por los profesionales del sector.